In [78]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [79]:
import os
import sys

import cv2

import numpy as np 

import torch
import torch.nn as nn
import torch.nn.functional as F

from box import Box
import yaml

import matplotlib.pyplot as plt

from scipy.spatial.transform import Rotation as R
import random 

import time 

root_dir = os.path.abspath('../..')
if root_dir not in sys.path:
    sys.path.append(root_dir)

from src.models.graph_inference import Graph as Graph_interference
from src.models.graph_training import Graph as Graph_train

# from src.data_loader.transforms_numpy import polar2cartesian, addSpeckleNoise, energyLoss, addBandReflects




- Create instances of graphs
- Load config files

In [80]:
model_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/model.yaml'
sonar_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/sonar.yaml'
train_config_pth = 'C:/Users/janis/Projekty/Magisterka/SonarOdometry/config/training.yaml'

with open(model_config_pth, "r") as f:
    model_config = Box(yaml.safe_load(f))

with open(sonar_config_pth, "r") as f:
    sonar_config = Box(yaml.safe_load(f))

with open(train_config_pth, "r") as f:
    train_config = Box(yaml.safe_load(f))


PatchGraph_i = Graph_interference(model_config, sonar_config)
PatchGraph_t = Graph_train(model_config, sonar_config, train_config)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

- prepare data 

In [81]:
# test data 

data_pth_sim = f'C:/Users/janis/Projekty/Magisterka/SonarOdometry/data/seq_1/fls/200.png'

# read frame 
frame_sim = cv2.imread(data_pth_sim, 0)

frame_sim = frame_sim.astype(np.uint8)
frame_sim = torch.from_numpy(frame_sim) # convert to torch tensor
frame_sim = frame_sim.float().unsqueeze(0).unsqueeze(0).unsqueeze(0)


frames_inm_series = 5
batch_size = 2
t_0 = 1.0
dt = 0.5

frame_sim_b = torch.cat([frame_sim for _ in range(frames_inm_series)], axis = 1)
frame_sim_b = torch.cat([frame_sim_b for _ in range(batch_size)], axis = 0)


time_tensor = torch.tensor([t_0 + i*dt for i in range(frames_inm_series)], device = device, dtype = torch.float)
time_tensor = time_tensor.unsqueeze(0)
time_tensor = torch.cat([time_tensor for _ in range(batch_size)], dim = 0)


print('-'*80)
print(f'Input data format:')
print(f'simulated tensor shape: {frame_sim.shape}, data type: {frame_sim.dtype}')
print(f'simulated tensors batch shape: {frame_sim_b.shape}, data type: {frame_sim_b.dtype}')
print(f'time tensor: {time_tensor.shape}')
print('-'*80)

--------------------------------------------------------------------------------
Input data format:
simulated tensor shape: torch.Size([1, 1, 1, 800, 768]), data type: torch.float32
simulated tensors batch shape: torch.Size([2, 5, 1, 800, 768]), data type: torch.float32
time tensor: torch.Size([2, 5])
--------------------------------------------------------------------------------


## Output test


In [82]:
def compare_shuffled(t1, t2):
    if t1.shape != t2.shape: return False
    # Spłaszczamy do 2D (N, -1) i sortujemy wiersze leksykograficznie
    t1_f = t1.reshape(t1.size(0), -1)
    t2_f = t2.reshape(t2.size(0), -1)
    for i in range(t1_f.shape[1]-1, -1, -1):
        t1_f = t1_f[t1_f[:, i].argsort(stable=True)]
        t2_f = t2_f[t2_f[:, i].argsort(stable=True)]
    return torch.allclose(t1_f.float(), t2_f.float(), atol=1e-3)

In [83]:
# === traning graph === 
t_start = time.time()

poses, coords_phi = PatchGraph_t.append(frame_sim_b, time_tensor, device)
corr_t, ctx_t, source_frame_idx_t, target_frame_idx_t, patch_idx_t = PatchGraph_t.update_step(poses, coords_phi,  device)

traning_time = time.time() - t_start

batch size: 2. frames: 5
debug: torch.Size([2, 5, 7])
debug: torch.Size([10, 7])


In [84]:
# === inference graph === 
t_start = time.time()

b, n, c, h, w = frame_sim_b.shape
print(frame_sim_b.shape)
for batch in range(b):
    for frame_num in range(n):
        frame = frame_sim_b[batch, frame_num, :, :, :].unsqueeze(0).unsqueeze(0)
        t = time_tensor[batch, frame_num]

        PatchGraph_i.append(frame, t, device)
        corr, ctx, source_frame_idx, target_frame_idx, patch_idx = PatchGraph_i.update_step(device)

corr_i, ctx_i, source_frame_idx_i, target_frame_idx_i, patch_idx_i = corr, ctx, source_frame_idx, target_frame_idx, patch_idx
inference_time = time.time() - t_start

    

torch.Size([2, 5, 1, 800, 768])


In [85]:
# comparision:
print('='*80)
print(f'Time: traning: {traning_time}, inference: {inference_time}')

print(f'number if edges in traning: {corr_t.shape}')
print(f'number if edges in inference: {corr_i.shape}')

output_test = []

output_test.append('Passed') if compare_shuffled(source_frame_idx_t, source_frame_idx_i) else output_test.append('Failed')
print(f'source_frame_idx test: {output_test[0]}')

output_test.append('Passed') if compare_shuffled(target_frame_idx_t, target_frame_idx_i) else output_test.append('Failed')
print(f'target_frame_idx test: {output_test[1]}')

output_test.append('Passed') if compare_shuffled(patch_idx_t, patch_idx_i) else output_test.append('Failed')
print(f'patch_idx test: {output_test[2]}')

Time: traning: 9.803590297698975, inference: 44.41348886489868
number if edges in traning: torch.Size([180, 50])
number if edges in inference: torch.Size([240, 50])
source_frame_idx test: Failed
target_frame_idx test: Failed
patch_idx test: Failed


In [86]:
# display edges:
print(PatchGraph_i.frame_n)
for a in range(PatchGraph_i.i.shape[0]):
    # if a > 90:
    print(PatchGraph_i.i[a], PatchGraph_i.j[a])

10
tensor(0) tensor(-1)
tensor(1) tensor(-1)
tensor(2) tensor(-1)
tensor(3) tensor(-1)
tensor(4) tensor(-1)
tensor(0) tensor(-2)
tensor(1) tensor(-2)
tensor(2) tensor(-2)
tensor(3) tensor(-2)
tensor(4) tensor(-2)
tensor(0) tensor(-3)
tensor(1) tensor(-3)
tensor(2) tensor(-3)
tensor(3) tensor(-3)
tensor(4) tensor(-3)
tensor(-15) tensor(0)
tensor(-14) tensor(0)
tensor(-13) tensor(0)
tensor(-12) tensor(0)
tensor(-11) tensor(0)
tensor(-10) tensor(0)
tensor(-9) tensor(0)
tensor(-8) tensor(0)
tensor(-7) tensor(0)
tensor(-6) tensor(0)
tensor(-5) tensor(0)
tensor(-4) tensor(0)
tensor(-3) tensor(0)
tensor(-2) tensor(0)
tensor(-1) tensor(0)
tensor(5) tensor(0)
tensor(6) tensor(0)
tensor(7) tensor(0)
tensor(8) tensor(0)
tensor(9) tensor(0)
tensor(5) tensor(-1)
tensor(6) tensor(-1)
tensor(7) tensor(-1)
tensor(8) tensor(-1)
tensor(9) tensor(-1)
tensor(5) tensor(-2)
tensor(6) tensor(-2)
tensor(7) tensor(-2)
tensor(8) tensor(-2)
tensor(9) tensor(-2)
tensor(-10) tensor(1)
tensor(-9) tensor(1)
tensor(-

In [ ]:
output = PatchGraph_i.visu_data(source_frame_idx_i, target_frame_idx_i, patch_idx_i)

for i, rec in enumerate(output):
    id, frames, coords = rec
    print(f'id: {id}, frames: {frames.shape}, coords: {coords.shape}')
    # print(f'Patch: {i}. Id: {id}. Occurance: {frames.shape[0]}')
    # for j in range(frames.shape[0]):
    #     print(f'    In frame: {frames[i]}, coords: {coords[i,:]}')

Patch: 0. Id: 35. Occurance: 3
    In frame: 7, coords: tensor([ 3.8434e+01, -1.1963e-02,  0.0000e+00])
    In frame: 7, coords: tensor([ 3.8434e+01, -1.1963e-02,  0.0000e+00])
    In frame: 7, coords: tensor([ 3.8434e+01, -1.1963e-02,  0.0000e+00])
Patch: 1. Id: 36. Occurance: 3
    In frame: 8, coords: tensor([1.7042e+01, 9.6938e-03, 0.0000e+00])
    In frame: 8, coords: tensor([1.7042e+01, 9.6938e-03, 0.0000e+00])
    In frame: 8, coords: tensor([1.7042e+01, 9.6938e-03, 0.0000e+00])
Patch: 2. Id: 37. Occurance: 3
    In frame: 9, coords: tensor([13.4350,  0.0185,  0.0000])
    In frame: 9, coords: tensor([13.4350,  0.0185,  0.0000])
    In frame: 9, coords: tensor([13.4350,  0.0185,  0.0000])
Patch: 3. Id: 38. Occurance: 3


IndexError: index 3 is out of bounds for dimension 0 with size 3